# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# Check data integration

In [0]:
TABLE_CONFIG = [
    "bronze.erp_px_cat_g1v2",
    "bronze.crm_prd_info",
    "bronze.crm_sales_details"
]

for table in TABLE_CONFIG:
    spark.read.table(table).limit(50).display()

# Read the dataset

In [0]:
df = spark.read.table("bronze.erp_px_cat_g1v2")

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Check is there any duplicate value and null value on ID column

In [0]:
df.groupBy("ID").agg(count("ID").alias("count_of_id")).filter((col("count_of_id") > 1) | (col("ID").isNull())).display()

# Check data standardization of CAT, SUBCAT, MAINTANANCE

In [0]:
COL = ["CAT","SUBCAT","MAINTENANCE"]

for column in COL:
    df.select(column).distinct().display()

# Normalize MAINTENANCE column into boolean flag

In [0]:
df = df.withColumn(
  "MAINTENANCE",
  when(col("MAINTENANCE") == "Yes", lit(True))
  .when(col("MAINTENANCE") == "No", lit(False))
  .otherwise(None)
)

# Sanity Check

In [0]:
df.display()